# Week 2 — Python fundamentals for toxicology data

**Goal:** be comfortable with the Python you'll use every day before we touch Tox21.

| Part | Topic | Why it matters for ML |
|---|---|---|
| 1 | Making decisions (`if` / `elif` / `else`) | Classifying chemicals, cleaning data |
| 2 | Loops (`for`, `while`, `enumerate`, `zip`) | Processing many chemicals |
| 3 | Comprehensions | One-line data transformations |
| 4 | Dictionaries in depth | Records, settings, lookups |
| 5 | Strings | SMILES, names, file paths |
| 6 | Functions that do real work | Reusable, testable steps |
| 7 | Errors and how to handle them | Messy real-world data |
| 8 | Reading and writing files | Getting data in and out |
| 9 | Mini-project | Putting it all together |

**How to use this notebook:** run each cell with **Shift+Enter**, read the output, then change something and run it again. Breaking things is how you learn. Exercises are marked ✏️; solutions are at the very end, so try first.

Kernel: **comptox** (top right).

## Part 1 — Making decisions

`if` runs code only when a condition is `True`. `elif` ("else if") tries the next condition; `else` catches everything left.

We'll use the **GHS acute aquatic toxicity categories**, based on the 96-hour fish LC50 (mg/L):

| LC50 (mg/L) | Category |
|---|---|
| ≤ 1 | Acute 1 (very toxic) |
| > 1 to ≤ 10 | Acute 2 (toxic) |
| > 10 to ≤ 100 | Acute 3 (harmful) |
| > 100 | Not classified |

In [ ]:
lc50 = 4.2   # mg/L

if lc50 <= 1:
    category = "Acute 1"
elif lc50 <= 10:      # only checked if the first condition was False
    category = "Acute 2"
elif lc50 <= 100:
    category = "Acute 3"
else:
    category = "Not classified"

print(f"LC50 {lc50} mg/L -> {category}")

**Comparison operators:** `==` equal, `!=` not equal, `<`, `<=`, `>`, `>=`
**Combining conditions:** `and`, `or`, `not`

Note `=` *assigns* a value and `==` *compares* — a very common early mistake.

In [ ]:
log_kow = 6.9          # octanol-water partition coefficient (log)
half_life_days = 400   # in sediment

# A rough persistence + bioaccumulation flag (illustrative thresholds)
bioaccumulative = log_kow >= 5
persistent = half_life_days > 180

if persistent and bioaccumulative:
    print("Flag: persistent AND bioaccumulative - candidate PBT")
elif persistent or bioaccumulative:
    print("Flag: one of P or B")
else:
    print("No P/B flag")

✏️ **Exercise 1.** Change `lc50` above to `0.5`, `50` and `250` and check each category is right. Then answer: what category does an LC50 of exactly `10` get, and why?

## Part 2 — Loops

A `for` loop repeats code once for each item in a collection.

In [ ]:
pollutants = ["PCB-153", "TCDD", "PFOA", "PFOS", "DDT"]

for name in pollutants:
    print("Checking", name)

**`enumerate`** gives you a counter alongside each item; **`zip`** walks through two lists side by side.

In [ ]:
lc50_values = [0.03, 0.0001, 12.5, 8.1, 0.9]   # mg/L, same order as `pollutants`

for i, name in enumerate(pollutants, start=1):
    print(i, name)

print()

for name, value in zip(pollutants, lc50_values):
    print(f"{name:10s} LC50 = {value} mg/L")

**`range`** produces numbers, handy for repeating something a set number of times.
**`break`** stops a loop early; **`continue`** skips to the next item.

In [ ]:
for fold in range(5):          # 0, 1, 2, 3, 4 - like 5-fold cross-validation
    print("fold", fold)

print()

for name, value in zip(pollutants, lc50_values):
    if value > 10:
        continue               # skip the less toxic ones
    print(name, "is Acute 1 or 2")

A `while` loop repeats *as long as* a condition is true. Useful when you don't know in advance how many steps you need.

In [ ]:
# How many half-lives until less than 1% of a pollutant remains?
remaining = 100.0   # percent
half_lives = 0

while remaining >= 1:
    remaining = remaining / 2
    half_lives += 1          # same as: half_lives = half_lives + 1

print(f"{half_lives} half-lives -> {remaining:.2f}% remaining")

✏️ **Exercise 2.** Using a `for` loop with `zip`, count how many pollutants in the lists above have an LC50 ≤ 1 mg/L. Print the count.

In [ ]:
# Exercise 2 - your code here

## Part 3 — Comprehensions

A **list comprehension** builds a new list in one readable line. You'll see these everywhere in data science code.

`[expression for item in collection if condition]`

In [ ]:
# The long way
ug_values = []
for v in lc50_values:
    ug_values.append(round(v * 1000, 3))   # mg/L -> ug/L

# The comprehension way - same result
ug_values = [round(v * 1000, 3) for v in lc50_values]
print("ug/L:", ug_values)

# With a filter
very_toxic = [name for name, v in zip(pollutants, lc50_values) if v <= 1]
print("Acute 1:", very_toxic)

A **dictionary comprehension** works the same way but builds key → value pairs.

In [ ]:
lc50_lookup = {name: v for name, v in zip(pollutants, lc50_values)}
print(lc50_lookup)
print("PFOA LC50:", lc50_lookup["PFOA"])

✏️ **Exercise 3.** Write one list comprehension that produces the names of pollutants whose LC50 is **greater than 1 but at most 10** mg/L.

In [ ]:
# Exercise 3 - your code here

## Part 4 — Dictionaries in depth

A dictionary maps **keys** to **values**. A list of dictionaries is the most natural way to hold records (one per chemical) before they become a pandas table.

In [ ]:
chemical = {
    "name": "PFOA",
    "cas": "335-67-1",
    "smiles": "OC(=O)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)F",
    "class": "PFAS",
    "lc50_mg_l": 12.5,
}

print(chemical["name"])
print(chemical.get("half_life"))              # .get returns None instead of crashing
print(chemical.get("half_life", "unknown"))   # ...or a default you choose

chemical["half_life_years"] = 3.8             # add a new key
print(list(chemical.keys()))

In [ ]:
# Looping over a dictionary
for key, value in chemical.items():
    print(f"{key:16s} {value}")

In [ ]:
# A list of records - this is exactly the shape pandas loves
records = [
    {"name": "PCB-153", "class": "PCB",    "lc50_mg_l": 0.03},
    {"name": "TCDD",    "class": "Dioxin", "lc50_mg_l": 0.0001},
    {"name": "PFOA",    "class": "PFAS",   "lc50_mg_l": 12.5},
    {"name": "PFOS",    "class": "PFAS",   "lc50_mg_l": 8.1},
]

# Group names by class
by_class = {}
for r in records:
    by_class.setdefault(r["class"], []).append(r["name"])
print(by_class)

✏️ **Exercise 4.** Using `records`, build a dictionary that maps each **class** to the **lowest** LC50 in that class (the most toxic member). Expected: `{'PCB': 0.03, 'Dioxin': 0.0001, 'PFAS': 8.1}`.

In [ ]:
# Exercise 4 - your code here

## Part 5 — Strings

Chemical data is full of text: names, CAS numbers, SMILES, file paths.

In [ ]:
smiles = "OC(=O)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)F"

print(len(smiles))                 # number of characters
print(smiles.count("F"))           # count fluorines (crudely - RDKit does this properly)
print("Cl" in smiles)              # membership test -> False
print(smiles[:6])                  # slicing: first 6 characters
print(smiles.lower()[:6])

In [ ]:
# Cleaning messy names - very common with real datasets
raw_names = ["  pfoa ", "PFOS\n", "Pcb-153", "tcdd"]
clean = [n.strip().upper() for n in raw_names]
print(clean)

# Splitting and joining
cas = "335-67-1"
parts = cas.split("-")
print(parts)
print("".join(parts))

In [ ]:
# f-strings: formatting numbers for reports
auc = 0.712345
p_value = 0.000031
print(f"AUC = {auc:.3f}")          # 3 decimal places
print(f"p = {p_value:.2e}")        # scientific notation
print(f"{'PFOA':>10}|")            # right-align in 10 characters

✏️ **Exercise 5.** A valid CAS number has three parts separated by hyphens, and the last part is a single digit. Write code that checks `cas = "335-67-1"` and prints `True` if it has that shape. Then try `"33567-1"` (should be `False`).

In [ ]:
# Exercise 5 - your code here

## Part 6 — Functions that do real work

Turn every repeated step into a function. Good functions have one job, a clear name, a docstring, and a `return` value.

In [ ]:
def ghs_aquatic_category(lc50_mg_l):
    '''Return the GHS acute aquatic category for a 96 h fish LC50 in mg/L.'''
    if lc50_mg_l <= 1:
        return "Acute 1"
    if lc50_mg_l <= 10:
        return "Acute 2"
    if lc50_mg_l <= 100:
        return "Acute 3"
    return "Not classified"


for r in records:
    print(r["name"], "->", ghs_aquatic_category(r["lc50_mg_l"]))

**Default arguments** and **keyword arguments** make functions flexible - this is exactly how scikit-learn models are configured.

In [ ]:
def convert_concentration(value, from_unit="mg/L", to_unit="ug/L"):
    '''Convert between mg/L, ug/L and ng/L.'''
    factors = {"mg/L": 1e-3, "ug/L": 1e-6, "ng/L": 1e-9}   # in g/L
    return value * factors[from_unit] / factors[to_unit]


print(convert_concentration(0.03))                          # mg/L -> ug/L
print(convert_concentration(70, from_unit="ng/L", to_unit="ug/L"))

Why `30.000000000000004` and not `30`? Computers store decimals in binary, so most can only be approximated - the same reason 1/3 can't be written exactly in decimal. It's not a bug in your code. Two habits follow:

- **Round when displaying:** `round(x, 6)` or `f"{x:.3f}"`
- **Never test decimals with `==`:** use `math.isclose(a, b)` instead

In [ ]:
import math

x = convert_concentration(0.03)
print(round(x, 6))
print(x == 30)               # False - surprising!
print(math.isclose(x, 30))   # True - the right way to compare decimals

A function can return several values at once (as a tuple), which you can unpack:

In [ ]:
def summarise(values):
    '''Return min, max and mean of a list of numbers.'''
    return min(values), max(values), sum(values) / len(values)


low, high, mean = summarise(lc50_values)
print(f"min {low}, max {high}, mean {mean:.3f}")

✏️ **Exercise 6.** Write a function `is_pbt_candidate(log_kow, half_life_days)` that returns `True` when `log_kow >= 5` **and** `half_life_days > 180`, otherwise `False`. Test it on `(6.9, 400)`, `(3.0, 400)` and `(6.9, 30)`.

In [ ]:
# Exercise 6 - your code here

## Part 7 — Errors and how to handle them

Real data is messy: missing values, text where numbers should be, typos. When Python hits a problem it raises an **exception**. Reading the last line of the error message tells you what went wrong.

In [ ]:
# Uncomment one line at a time to see each error, then comment it again
# float("not reported")        # ValueError
# lc50_lookup["Mercury"]       # KeyError
# 10 / 0                       # ZeroDivisionError
print("Uncomment a line above to see an error")

`try` / `except` lets you handle a problem instead of crashing. Catch **specific** errors, and never hide problems silently in a scientific pipeline - record them.

In [ ]:
raw_lc50 = ["0.03", "0.0001", "not reported", "8.1", "", ">100"]

clean_values = []
problems = []

for i, text in enumerate(raw_lc50):
    try:
        clean_values.append(float(text))
    except ValueError:
        problems.append((i, text))      # keep a record - don't hide it

print("Parsed:", clean_values)
print("Could not parse:", problems)

Notice `">100"` failed. In toxicology that is a **censored value** (the true LC50 is above the highest dose tested) - not junk. How you handle it is a scientific decision, which is why you record problems instead of silently dropping them.

✏️ **Exercise 7.** Write a function `parse_lc50(text)` that returns a float for normal numbers, returns `None` for empty or "not reported" values, and for values starting with `">"` returns the number after the `>` (we'll treat it as a lower bound). Test it on every item in `raw_lc50`.

In [ ]:
# Exercise 7 - your code here

## Part 8 — Reading and writing files

The `with` statement opens a file and closes it automatically. The `csv` module reads and writes tables. (Next week, pandas will do this in one line - but it helps to know what's underneath.)

In [ ]:
import csv
from pathlib import Path

out = Path("pollutants_week02.csv")

with open(out, "w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=["name", "class", "lc50_mg_l"])
    writer.writeheader()
    writer.writerows(records)

print("Wrote", out.resolve())

In [ ]:
with open(out, newline="") as fh:
    rows = list(csv.DictReader(fh))

for row in rows:
    print(row)

# Everything read from a CSV is text - convert numbers yourself
print(type(rows[0]["lc50_mg_l"]))

💡 Add `*.csv` outputs like this to `.gitignore` if they're scratch files. Your repo's `.gitignore` already excludes `data/raw/`.

## Part 9 — Mini-project: a hazard screening report

Combine everything: start from messy raw records, clean them, classify them, and print a short report.

In [ ]:
raw_records = [
    {"name": " pcb-153 ", "class": "PCB",    "lc50": "0.03",  "log_kow": "6.9", "half_life_days": "1000"},
    {"name": "TCDD",      "class": "Dioxin", "lc50": "0.0001","log_kow": "6.8", "half_life_days": "3650"},
    {"name": "pfoa",      "class": "PFAS",   "lc50": "12.5",  "log_kow": "",    "half_life_days": "1400"},
    {"name": "PFOS\n",    "class": "PFAS",   "lc50": "8.1",   "log_kow": "",    "half_life_days": "1500"},
    {"name": "Ethanol",   "class": "Solvent","lc50": ">100",  "log_kow": "-0.3","half_life_days": "5"},
    {"name": "Mystery X", "class": "Unknown","lc50": "not reported", "log_kow": "4.1", "half_life_days": ""},
]


def to_float(text):
    '''Float, or None for blank / unparseable values.'''
    text = text.strip().lstrip(">")
    try:
        return float(text)
    except ValueError:
        return None


report = []
for r in raw_records:
    name = r["name"].strip().upper()
    lc50 = to_float(r["lc50"])
    censored = r["lc50"].strip().startswith(">")   # true LC50 is ABOVE this value
    log_kow = to_float(r["log_kow"])
    half_life = to_float(r["half_life_days"])

    if lc50 is None:
        category = "No data"
    elif censored and lc50 >= 100:
        category = "Not classified"   # above 100 mg/L, whatever the exact value
    elif censored:
        category = "Censored - check" # e.g. ">5": could be Acute 2 or less toxic
    else:
        category = ghs_aquatic_category(lc50)
    if log_kow is None or half_life is None:
        pbt = "Insufficient data"
    else:
        pbt = "Yes" if (log_kow >= 5 and half_life > 180) else "No"

    report.append({"name": name, "category": category, "pbt": pbt})

print(f"{'Chemical':12s} {'GHS aquatic':16s} {'PBT?'}")
print("-" * 45)
for row in report:
    print(f"{row['name']:12s} {row['category']:16s} {row['pbt']}")

**Things to notice**

- PFOA and PFOS show *Insufficient data* for PBT because log Kow is blank - and log Kow is a poor predictor for PFAS anyway (they bind proteins rather than partition into fat). A screening rule is only as good as its assumptions.
- Ethanol's `>100` means the true LC50 is *above* 100 mg/L, so it is **Not classified**. Treating it as exactly 100 would wrongly make it "Acute 3" - censored values need their own rule. A value like `>5` can't be classified at all from that number alone, so the code flags it rather than guessing.
- "Mystery X" is not silently dropped; it's reported as *No data*.

These are the same habits your validator enforces: **never pass when blind, never hide a problem.**

✏️ **Exercise 8 (stretch).** Extend the report with a column `notes` that says `"censored LC50"` whenever the raw LC50 started with `">"`. Then add a record with an LC50 of `">5"` to `raw_records` and check it comes out as *Censored - check*.

In [ ]:
# Exercise 8 - your code here

---
## Solutions

Try each exercise before looking. There is usually more than one correct answer.

**Exercise 1.** An LC50 of exactly `10` gives **Acute 2**: the first condition (`<= 1`) is False, the second (`<= 10`) is True, so Python stops there. `elif` branches are checked in order and only the first True one runs.

In [ ]:
# Exercise 2
count = 0
for name, value in zip(pollutants, lc50_values):
    if value <= 1:
        count += 1
print("Acute 1 count:", count)

In [ ]:
# Exercise 3
acute2 = [name for name, v in zip(pollutants, lc50_values) if 1 < v <= 10]
print(acute2)

In [ ]:
# Exercise 4
lowest = {}
for r in records:
    c, v = r["class"], r["lc50_mg_l"]
    if c not in lowest or v < lowest[c]:
        lowest[c] = v
print(lowest)

In [ ]:
# Exercise 5
def looks_like_cas(cas):
    parts = cas.split("-")
    return len(parts) == 3 and all(p.isdigit() for p in parts) and len(parts[2]) == 1

print(looks_like_cas("335-67-1"), looks_like_cas("33567-1"))

In [ ]:
# Exercise 6
def is_pbt_candidate(log_kow, half_life_days):
    return log_kow >= 5 and half_life_days > 180

print(is_pbt_candidate(6.9, 400), is_pbt_candidate(3.0, 400), is_pbt_candidate(6.9, 30))

In [ ]:
# Exercise 7
def parse_lc50(text):
    text = text.strip()
    if text == "" or text.lower() == "not reported":
        return None
    if text.startswith(">"):
        return float(text[1:])
    return float(text)

print([parse_lc50(t) for t in raw_lc50])

In [ ]:
# Exercise 8
for r, row in zip(raw_records, report):
    row["notes"] = "censored LC50" if r["lc50"].strip().startswith(">") else ""

for row in report:
    print(f"{row['name']:12s} {row['category']:16s} {row['pbt']:18s} {row['notes']}")

# Second part: in the Part 9 cell, add this line inside raw_records, re-run that
# cell, then re-run this one:
#   {"name": "Test", "class": "Test", "lc50": ">5", "log_kow": "", "half_life_days": ""},

---
### Next week

**NumPy and pandas**: loading a real toxicity table in one line, filtering, grouping and plotting - the bridge to the Tox21 classifier.

Remember to commit:
```
git add .
git commit -m "Week 2: Python fundamentals"
git push
```